# Model 11

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"     # agama
os.environ["MKL_NUM_THREADS"] = "1"     # numpy, scipy
os.environ["OPENBLAS_NUM_THREADS"] = "1"    # numpy
os.environ["NUMEXPR_NUM_THREADS"] = "1"     # pandas

import agama
import torch 
import numpy as np
from scipy import integrate
from scipy.stats import wasserstein_distance
from astropy import units as u

from sbi.utils import BoxUniform
from sbi.inference import SNLE, simulate_for_sbi, prepare_for_sbi
from sbi.utils import likelihood_nn

from sklearn.metrics import mean_squared_error, r2_score

import pandas as pd
import pickle
import matplotlib.pyplot as plt
from galaxy_generation import generate_galaxy_multiple
from prior_generation import generate_prior

import corner

torch.set_num_threads(1)


In [2]:
# set agama unit to be in Msun, kpc, km/s
agama.setUnits(mass=1 * u.Msun, length=1*u.kpc, velocity=1 * u.km /u.s)
agama.setRandomSeed(13)
torch.manual_seed(13)
np.random.seed(13)


## Generate Galaxy

In [5]:
num_galaxies = 1000
prior = generate_prior()
theta = prior.sample((num_galaxies,))
n_stars = np.random.poisson(100, size=num_galaxies)

theta = torch.repeat_interleave(theta, torch.tensor(n_stars), dim=0)
x = generate_galaxy_multiple(theta, n_stars, 25)

In [6]:
pd.DataFrame(x).to_csv("test_x.csv", index=None, header=None)
pd.DataFrame(theta).to_csv("test_theta.csv", index=None, header=None)

## Contour simulations

In [ ]:
with open("inference_model_11.pkl", "rb") as file:
    inference = pickle.load(file)

cored, gamma = 0

In [ ]:
galaxy = np.expand_dims(np.array([7, 0, 0, 0.2]), axis=0).astype(np.float32)
df = inference._neural_net.sample(100_000, context=galaxy).squeeze().detach().numpy()
pd.DataFrame(df).to_csv("contour_samples_model_11_core.csv", header=None, index=None)

cuspy, gamma =1

In [ ]:
galaxy = np.expand_dims(np.array([7, 0, 1, 0.2]), axis=0).astype(np.float32)
df = inference._neural_net.sample(100_000, context=galaxy).squeeze().detach().numpy()
pd.DataFrame(df).to_csv("contour_samples_model_11_cusp.csv", header=None, index=None)


### Optuna

In [9]:
with open("tune_model_11.pkl", "rb") as file:
    o = pickle.load(file)

### Sequential x_o

In [18]:
df = np.array(pd.read_csv("x_o_cusp.csv", header=None))

In [19]:
l = np.zeros((df.shape[0], 2))
l[:, 0] = np.sqrt(df[:,0] ** 2 + df[:,1] ** 2)
l[:, 1] = df[:, -1]

In [20]:
pd.DataFrame(l).to_csv("x_o_cusp.csv", header=None, index=None)